
# Mixing elbow in resilient mode
This example sets up and solves the mixing elbow with ordinary settings API
code, launched with ``resilient_mode=True``. It shows how resilient mode:

* undoes and redoes settings writes,
* checkpoints before ``iterate`` and rolls back on request.

See `ref_resilient_guide` for details.


## Perform required imports



In [ ]:
import ansys.fluent.core as pyfluent
from ansys.fluent.core import examples

import_file_name = examples.download_file(
    "mixing_elbow.msh.h5", "pyfluent/mixing_elbow"
)

## Launch Fluent in resilient mode



In [ ]:
solver_session = pyfluent.Solver.from_install(
    precision=pyfluent.Precision.DOUBLE,
    processor_count=2,
    resilient_mode=True,
)
resilient = solver_session.resilient
resilient.on_event(lambda event: print(f"[{event.kind.value}] {event.message}"))

## Read the mesh



In [ ]:
solver_session.settings.file.read_mesh(file_name=import_file_name)

## Set up boundary conditions



In [ ]:
# Built-in solver objects:
energy = pyfluent.solver.Energy(settings_source=solver_session)
cold_inlet = pyfluent.solver.VelocityInlet(
    settings_source=solver_session, name="cold-inlet"
)
hot_inlet = pyfluent.solver.VelocityInlet(
    settings_source=solver_session, name="hot-inlet"
)
# Aliasing reused objects:
cold_turbulence = cold_inlet.turbulence
turbulent_intensity = cold_turbulence.turbulent_intensity
hot_turbulence = hot_inlet.turbulence

energy.enabled.set_state(True)
cold_inlet.momentum.velocity_magnitude.value.set_state(0.4)
cold_turbulence.turbulence_specification.set_state("Intensity and Hydraulic Diameter")
turbulent_intensity.set_state(0.05)
cold_turbulence.hydraulic_diameter.set_state("4 [in]")
cold_inlet.thermal.temperature.value.set_state(293.15)

hot_inlet.momentum.velocity_magnitude.value.set_state(1.2)
hot_turbulence.turbulence_specification.set_state("Intensity and Hydraulic Diameter")
hot_turbulence.hydraulic_diameter.set_state("1 [in]")
hot_inlet.thermal.temperature.value.set_state(313.15)

## Undo and redo a settings change
Try a higher turbulent intensity at the cold inlet. Undo restores the 0.05 set
above, and redo applies 0.1 again. A second undo keeps the setup at 0.05 for
the solve.



In [ ]:
turbulent_intensity.set_state(0.1)
print("changed:", turbulent_intensity())
resilient.undo()
print("after undo:", turbulent_intensity())
resilient.redo()
print("after redo:", turbulent_intensity())
resilient.undo()
print("after a second undo:", turbulent_intensity())

## Initialize and checkpoint



In [ ]:
initialization = pyfluent.solver.Initialization(settings_source=solver_session)
initialization.hybrid_initialize()
resilient.checkpoint("initialized")

## Solve
``iterate`` is irreversible, so resilient mode writes an automatic checkpoint
before it runs. If the solve failed, the session would be restored to that
checkpoint and the error raised.



In [ ]:
run_calculation = pyfluent.solver.RunCalculation(settings_source=solver_session)
run_calculation.iterate(iter_count=100)
print("checkpoints:", [checkpoint.name for checkpoint in resilient.checkpoints])

## Roll back to the initialized state



In [ ]:
resilient.rollback("initialized")

## Exit



In [ ]:
solver_session.exit()